# പാഠം 09 - മെറ്റാകോഗ്നിഷൻ ഡിസൈൻ പാറ്റേൺ


## ക്രമീകരിക്കൽ

ഈ നോട്ട്‌ബുക്ക് Microsoft Agent ഫ്രെയിംവർക്ക് ഉപയോഗിച്ച് Metacognition ഡിസൈൻ പാറ്റേൺ പ്രദർശിപ്പിക്കുന്നു.

**ആവശ്യമായ കാര്യങ്ങൾ:**
- പരിസര വ്യത്യാസങ്ങൾ വഴി ക്രമീകരിച്ച Azure OpenAI ഡിപ്ലോയ്‌മെന്റ്
- Azure CLI പ്രാമാണീകരിച്ചത് (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## മെറ്റകോഗ്നിഷൻ എന്താണ്?

മെറ്റകോഗ്നിഷൻ എന്നത് **ചിന്തിക്കുന്നതിനെപ്പറ്റി ചിന്തിക്കുന്നത്** ആണ്. AI ഏജന്റുകളുടെ സാഹചര്യത്തിൽ, ഇതിനുർത്ഥം എജന്റുകൾക്ക് താഴെപ്പറയുന്നവ കഴിവുണ്ടാകണം എന്നാണ്:

- അവരുടെ സ്വന്തം ഔട്ട്പുട്ടുകളെയും നിരൂപണ പ്രക്രിയയെയും **സ്വയം പ്രതിബിംബിപ്പിക്കുക**
- **തെറ്റുകൾ കണ്ടെത്തി** ശബ്ദരഹിതമായി പരാജയപ്പെടുന്നതിനുപകരം എളുപ്പത്തിൽ മടക്കി വീണ്ടെടുക്കുക
- അവരുടെ പ്രതികരണങ്ങൾ പൂര്‍ണ്ണവും സഹായകരവുമാണോയെന്ന് **മൂല്യമാപനം ചെയ്യുക**
- തുടക്കത്തിലുള്ള സമീപനം പണിയാതിരുന്നാൽ തന്ത്രം **അനുയോജിപ്പിക്കുക** (ഉദാഹരണത്തിന്, ബാക്കപ്പ് സിസ്റ്റത്തിലേക്ക് തിരിച്ചു പോവുക)

ഒരു മെറ്റകോഗ്നിറ്റീവ് ഏജന്റ് വെറും ചോദ്യങ്ങൾക്കു മാത്രം മറുപടി നൽകുന്നില്ല - അത് തന്റെ പ്രകടനം നിരീക്ഷിക്കുകയും തദനുസരിച്ച് ഉടൻ ക്രമീകരിക്കുകയും ചെയ്യുന്നു.


## പ്രാഥമികവും ബാക്കപ്പ് ഉപകരണങ്ങളും

പൊതുവായ ഒരു മെറ്റാകോഗ്‌നിഷൻ മാതൃകയാണ് **ഫാല്ബാക്ക് സ്റ്റ്രാറ്റജി**. ഏജന്റ് ആദ്യം പ്രാഥമിക ഉപകരണം പരീക്ഷിക്കുന്നു; പരാജയപ്പെട്ടു എങ്കിൽ (ഉദാഹരണത്തിന്, 404 പിഴവ്), ഏജന്റ് പരാജയം തിരിച്ചറിയുകയും തുറന്ന മനസ്സോടെ ബാക്കപ്പ് ഉപകരണത്തിലേക്ക് മാറുകയും ചെയ്യുന്നു.

ഇത് യാഥാർത്ഥ്യത്തിൽ, പ്രാഥമിക സേവനങ്ങൾ ലഭ്യമല്ലാത്ത സമയങ്ങളിൽ ഏജന്റുകൾ സ്വയം പ്രശ്നം തിരിച്ചറിയുകയും മറ്റൊരു മാർഗം തിരഞ്ഞെടുക്കുകയും ചെയ്യുന്ന സിസ്റ്റങ്ങളോട് ഇലംപിക്കുന്നു.

താഴെ, ഞങ്ങൾ രണ്ടു വിമാന ലഭ്യത പരിശോധിക്കൽ ഉപകരണങ്ങൾ നിർവചിക്കുന്നു:
- **പ്രാഥമികം** — പാരീസ്, ടോക്കിയോ, ബാർസലോണ എന്നിവിടങ്ങളെ ഉൾക്കൊള്ളുന്നു
- **ബാക്കപ്പ്** — ബെർലിൻ, സിഡ്നി, ന്യൂയോർക്കു സിറ്റി എന്നിവിടങ്ങളെ ഉൾക്കൊള്ളുന്നു


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## പിശക് വീണ്ടെടുക്കലോടെ സ്വയം-പ്രതിബിംബിക്കുന്ന ഏജന്റ്

താഴെയുള്ള ഏജന്റിന് ആദ്യമേ പ്രധാന മാറ്റിവേലിനുള്ള സംവിധാനത്തെ പരീക്ഷിക്കുകയും, പരാജയങ്ങളെ തിരിച്ചറിയുകയും, പ്രസ്തുത പരാജയത്തിൽ പരദർശകമായി ബാക്കപ്പ് സംവിധാനത്തിലേക്ക് മടങ്ങി പോകുകയും ചെയ്യേണ്ടതാണ്. ഓരോ പ്രതികരണത്തിനും ശേഷം അത് സ്വയംക്ഷണമായി വിലയിരുത്തി ഉപയോഗകർത്താവിന്റെ ചോദ്യത്തിന് പൂർണ്ണമായും മറുപടി നൽകിയിട്ടുണ്ടോ എന്ന് പരിശോധിക്കുന്നു.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## സ്വയം മൂല്യനിർണയ പാറ്റേൺ

മിടാകോഗ്നിഷന്റെ മറ്റൊരു വശം **സ്വയം മൂല്യനിർണയമാണ്**: ഒരു വേര്‍പാട് ഏജൻറ് (അല്ലെങ്കിൽ മടങ്ങി പരിശോധിച്ചാൽ ഒരേ ഏജൻറ്) ഒരു പ്രതികരണത്തിന്റെ പൂർണത, കൃത്യത, സഹായക്ഷമത എന്നിവ വിലയിരുത്തുന്നു.

താഴെ നമ്മൾ ഒരു `ResponseEvaluator` ഏജൻറ് സൃഷ്ടിക്കുന്നു, ഇത് യാത്ര ഏജൻസി മറുപടികളെ മൂന്ന് തലങ്ങളിൽ സ്‌കോർ ചെയ്യുന്നു.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## സംഗ്രഹം

ഈ പാഠത്തിൽ നിങ്ങൾ Microsoft Agent Framework ഉപയോഗിച്ച് **മെറ്റാക്‌നൊഗ്നിറ്റീവ് ഏജന്റുകൾ** എങ്ങനെ നിർമ്മിക്കാമെന്ന് പഠിച്ചു:

- **സ്വയംപരിശോധന**: സ്വയം തർക്കം നിരീക്ഷിക്കുന്നതും സംഭവിച്ചതിനെ താരതമ്യത്തിലാക്കിയും പ്രവർത്തിക്കുന്ന ഏജന്റുകൾ.
- **പിഴവു പരിഹാരം ഫാൾബാക്കുമായി**: ഒരു പ്രാഥമിക + ബാക്കപ്പ് ഉപകരണ മാതൃക, ഏജന്റിന് പിഴവുകൾ (ഉദാ., 404 പിഴവുകൾ) കണ്ടെത്തുകയും സ്വയമേവ മറ്റൊരു ഉറവിടം പരീക്ഷിക്കുകയും ചെയ്യുന്നത്.
- **സ്വയംമൂല്യനിർണ്ണയം**: പൂർണ്ണത, കൃത്യത, സഹായകത എന്നിവയ്ക്ക് പ്രതികരണങ്ങളെ സ്കോർ ചെയ്യുന്ന പ്രത്യേക മൂല്യനിർണായക ഏജന്റ്.

ഈ മാതൃകകൾ ഏജന്റുകൾക്ക് കൂടുതൽ ദൃഢത,താരതമ്യം, വിശ്വാസ്യത എന്നിവ നൽകുന്നു — ഉത്പാദന വിന്യാസങ്ങൾക്ക് അത്യന്താപേക്ഷിത ഗുണലക്ഷണങ്ങൾ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
